# 04 — Agent 2: Conversion Predictor (LightGBM + SMOTE)

**Model**: LightGBM with SMOTE oversampling for 22% class imbalance
**Target**: Policy_Bind (Yes=1/No=0)
**Output**: bind_score (0-100)
**Explainability**: SHAP + LIME + Anchors + DiCE

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

MODELS_DIR = Path("../models")
train_df = pd.read_parquet(MODELS_DIR / "train.parquet")
test_df = pd.read_parquet(MODELS_DIR / "test.parquet")
feature_config = joblib.load(MODELS_DIR / "feature_config.joblib")

FEATURES = feature_config["AGENT2_FEATURES"]
TARGET = "Policy_Bind_enc"

X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()
X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\nClass imbalance (train):")
print(y_train.value_counts())
print(f"\nBind rate: {y_train.mean():.4f} ({y_train.mean()*100:.1f}%)")

## 1. SMOTE Oversampling
Apply SMOTE only to training set. Test set remains at real-world 22% distribution.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {X_train.shape[0]} rows")
print(f"After SMOTE:  {X_train_sm.shape[0]} rows")
print(f"\nResampled distribution:")
print(pd.Series(y_train_sm).value_counts())

## 2. Train LightGBM

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

model = LGBMClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=50,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

model.fit(X_train_sm, y_train_sm, eval_set=[(X_test, y_test)])

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["No Bind", "Bind"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob):.4f}")

## 3. Evaluation Plots

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No Bind", "Bind"],
            yticklabels=["No Bind", "Bind"], ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1], name="LightGBM")
axes[1].set_title("ROC Curve")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)

PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[2], name="LightGBM")
axes[2].set_title("Precision-Recall Curve")

plt.tight_layout()
plt.savefig(MODELS_DIR / "agent2_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Bind Score Distribution

In [ ]:
bind_scores = (y_prob * 100).astype(int)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(bind_scores[y_test == 0], bins=50, alpha=0.6, color="red", label="Did NOT bind")
ax.hist(bind_scores[y_test == 1], bins=50, alpha=0.6, color="green", label="Did bind")
ax.set_xlabel("Bind Score (0-100)")
ax.set_ylabel("Count")
ax.set_title("Bind Score Distribution")
ax.legend()
plt.tight_layout()
plt.savefig(MODELS_DIR / "agent2_bind_score_dist.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean bind score — Did NOT bind: {bind_scores[y_test == 0].mean():.1f}")
print(f"Mean bind score — Did bind:     {bind_scores[y_test == 1].mean():.1f}")

## 5. Explainability: SHAP

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
X_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_sample)

if isinstance(shap_values, list):
    shap_bind = shap_values[1]
else:
    shap_bind = shap_values

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_bind, X_sample, show=False)
plt.title("SHAP — Feature Importance for Bind Prediction")
plt.tight_layout()
plt.savefig(MODELS_DIR / "agent2_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

sample_row = X_test.iloc[[0]]
prob = model.predict_proba(sample_row)[0, 1]
sv_single = explainer.shap_values(sample_row)
if isinstance(sv_single, list):
    bind_shap = sv_single[1][0]
else:
    bind_shap = sv_single[0]
paired = sorted(zip(FEATURES, bind_shap), key=lambda p: abs(p[1]), reverse=True)
print(f"\nBind probability: {prob:.4f} (score: {int(prob*100)})")
print("Top SHAP features:")
for name, val in paired[:5]:
    print(f"  {name}: {val:+.4f}")

## 6. Explainability: LIME

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    X_train_sm.values,
    feature_names=FEATURES,
    class_names=["No Bind", "Bind"],
    mode="classification",
    random_state=42,
)

sample_idx = 0
lime_exp = lime_explainer.explain_instance(
    X_test.iloc[sample_idx].values,
    model.predict_proba,
    num_features=8,
    top_labels=1,
)

pred_prob = model.predict_proba(X_test.iloc[[sample_idx]])[0, 1]
available_labels = list(lime_exp.local_exp.keys())
label_to_show = 1 if 1 in available_labels else available_labels[0]
print(f"Bind probability: {pred_prob:.4f}")
print(f"\nLIME explanation:")
for feat, weight in lime_exp.as_list(label=label_to_show):
    print(f"  {feat}: {weight:+.4f}")

fig = lime_exp.as_pyplot_figure(label=label_to_show)
plt.title(f"LIME — Bind Score: {int(pred_prob*100)}")
plt.tight_layout()
plt.savefig(MODELS_DIR / "agent2_lime_example.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Explainability: Anchors (IF-THEN Rules)

In [ ]:
from alibi.explainers import AnchorTabular

anchor_explainer = AnchorTabular(
    predictor=model.predict,
    feature_names=FEATURES,
)
anchor_explainer.fit(X_train_sm.values, disc_perc=(25, 50, 75))

sample_idx = 0
explanation = anchor_explainer.explain(X_test.iloc[sample_idx].values)

pred = model.predict(X_test.iloc[[sample_idx]])[0]
anchor_rule = " AND ".join(explanation.anchor) if explanation.anchor else "No stable rule found"
print(f"Predicted: {'Bind' if pred == 1 else 'No Bind'}")
print(f"\nAnchor Rule: {anchor_rule}")
print(f"Precision: {explanation.precision:.4f}")
print(f"Coverage:  {explanation.coverage:.4f}")

## 8. Explainability: DiCE Counterfactuals

"What would need to change for this quote to convert?" — the most actionable insight for agents.

In [ ]:
import dice_ml

train_dice = X_train.copy()
train_dice["Policy_Bind_enc"] = y_train.values

continuous_features = [
    "urgency_days",
    "HH_Drivers",
    "HH_Vehicles",
    "Quoted_Premium",
    "Driver_Age",
    "Driving_Exp",
    "Prev_Accidents",
    "Prev_Citations",
]

d = dice_ml.Data(
    dataframe=train_dice,
    continuous_features=continuous_features,
    outcome_name="Policy_Bind_enc",
)

m = dice_ml.Model(model=model, backend="sklearn", model_type="classifier")
dice_exp = dice_ml.Dice(d, m, method="random")

sample_row = X_test.iloc[[0]]
pred = model.predict(sample_row)[0]

desired = 1 if pred == 0 else 0
print(f"Original: {'Bind' if pred == 1 else 'No Bind'} (score: {int(model.predict_proba(sample_row)[0,1]*100)})")
print(f"Desired: {'Bind' if desired == 1 else 'No Bind'}")
try:
    cf = dice_exp.generate_counterfactuals(
        sample_row,
        total_CFs=3,
        desired_class=desired,
    )
    print(f"\nCounterfactual examples:")
    cf.visualize_as_dataframe(show_only_changes=True)
except Exception as e:
    print(f"DiCE could not generate counterfactuals for this sample: {e}")

## 9. Save Model & Explainers

In [ ]:
joblib.dump(model, MODELS_DIR / "conversion_model.joblib")
joblib.dump(explainer, MODELS_DIR / "conversion_explainer.joblib")
print("Saved: conversion_model.joblib, conversion_explainer.joblib")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob):.4f}")